# ООП на практике


Объектно-ориентированное программирование помогает собрать связанные данные - **объекты**, и действия - **методы**, в одном месте.

Зачем нужны объекты?

Представим программу для учета. Допустим, мы храним расходы в словарях:

In [ ]:
expense_1 = {
    "title": "Кофе",
    "amount": 8.5,
    "category": "Еда",
}

expense_2 = {
    "title": "Такси",
    "amount": 24.0,
    "category": "Транспорт",
}

print(expense_1)


Словарь подходит для небольшой задачи. Но в большой программе легко:

* ошибиться в имени ключа;
* забыть обязательное поле;
* положить неверное значение;
* разбросать функции для работы с расходами по всему проекту.


In [ ]:
def describe_expense(expense: dict) -> str:
    return f'{expense["title"]}: {expense["amount"]:.2f} BYN'


print(describe_expense(expense_1))


Мы хотим явно описать новый тип данных `Expense`: какие данные он хранит и что умеет делать.

Для этого создадим **класс**.


## Класс и объект


**Класс** — описание объектов определённого вида.

**Объект** — конкретное значение, созданное по этому описанию.

Например, `str` — класс, а строки `"кофе"` и `"такси"` — два разных объекта этого класса.


In [ ]:
print(type("кофе"))
print(type(42))
print(type([1, 2, 3]))


Создадим самый простой собственный класс. Имена классов записывают в `PascalCase`.


In [ ]:
class Expense:
    pass


first_expense = Expense()
second_expense = Expense()

print(type(first_expense))
print(first_expense is second_expense)


`Expense()` создаёт новый объект. Два вызова создают два независимых объекта.

Пока класс пустой. Добавим объектам атрибуты — переменные, которые принадлежат конкретному объекту.


In [ ]:
first_expense.title = "Кофе"
first_expense.amount = 8.5

second_expense.title = "Такси"
second_expense.amount = 24.0

print(first_expense.title, first_expense.amount)
print(second_expense.title, second_expense.amount)


## Методы и self


Метод — функция, объявленная внутри класса.

Как одна функция понимает, с каким конкретно объектом она сейчас работает?

Объект передаётся методу первым аргументом метода. Этот параметр принято называть `self`.


In [ ]:
class Expense:
    def show(self):
        print("Текущий объект:", self)


coffee = Expense()
taxi = Expense()

print(coffee)
print(taxi)

coffee.show()
taxi.show()


В выражении `coffee.show()` Python выполняет два действия:

1. Находит метод `show` в классе `Expense`.
2. Автоматически передаёт `coffee` в первый параметр метода.

Поэтому эти два вызова равнозначны:

```python
coffee.show()
Expense.show(coffee)
```


In [ ]:
coffee.show()
Expense.show(coffee)


Это основная идея `self`:

> `self` — обычный параметр, в который Python передаёт объект слева от точки.

В вызове `coffee.show()` параметр `self` получит объект `coffee`.

В вызове `taxi.show()` тот же параметр получит объект `taxi`.


In [ ]:
class Expense:
    def remember_self(self):
        print("id(self):", id(self))


coffee = Expense()

print("id(coffee):", id(coffee))
coffee.remember_self()


Числа совпадают, потому что `self` и `coffee` указывают на один и тот же объект.

Имя `self` не является специальным словом Python. Технически параметр можно назвать иначе, но так делать не принято.


In [ ]:
class StrangeExample:
    def show_object(any_name):
        print(any_name)


example = StrangeExample()
example.show_object()


Всегда используйте имя `self`. Тогда любой Python-разработчик сразу узнает первый параметр метода.

Посмотрим, зачем `self` нужен на практике. Через него метод получает доступ к атрибутам **текущего** объекта.


In [ ]:
class Expense:
    def show(self):
        print(f"{self.title}: {self.amount:.2f} BYN")


coffee = Expense()
coffee.title = "Кофе"
coffee.amount = 8.5

taxi = Expense()
taxi.title = "Такси"
taxi.amount = 24.0

coffee.show()
taxi.show()


Метод `show` один, но результат разный:

* при `coffee.show()` выражение `self.title` означает `coffee.title`;
* при `taxi.show()` выражение `self.title` означает `taxi.title`.

Метод не хранит отдельные значения. Он получает объект через `self` и работает с его состоянием.

Вызов удобно читать слева направо:

```python
объект.метод(остальные_аргументы)
```

Объект слева от точки попадёт в `self`. Остальные аргументы распределятся по оставшимся параметрам.


In [ ]:
class Counter:
    def increase(self, step):
        self.value += step


counter = Counter()
counter.value = 10

counter.increase(3)
# То же самое: Counter.increase(counter, 3)

print(counter.value)


Здесь Python распределяет значения так:

| Параметр | Значение |
|---|---|
| `self` | `counter` |
| `step` | `3` |

Аргумент для `self` уже находится слева от точки, поэтому отдельно его не передаём.

Если метод объявлен без первого параметра, Python всё равно попытается передать объект. Получится лишний аргумент.


In [ ]:
class BrokenCounter:
    def increase():
        print("Увеличиваем")


broken = BrokenCounter()

try:
    broken.increase()
except TypeError as error:
    print(type(error))
    print(error)


Никогда не забывайте про `self`!

Частая ошибка -  забыли `self.` перед атрибутом

`amount` и `self.amount` — разные выражения.

* `amount` ищет локальную или глобальную переменную;
* `self.amount` берёт атрибут текущего объекта.


In [ ]:
amount = 1000


class Expense:
    def show_wrong(self):
        print("Глобальная переменная:", amount)

    def show_right(self):
        print("Атрибут объекта:", self.amount)


coffee = Expense()
coffee.amount = 8.5
coffee.show_wrong()
coffee.show_right()


Посмотрим пример ниже:

1. Что попадёт в `self` при вызове `red.paint("стена")`?
2. Что попадёт в `surface`?


In [ ]:
class Color:
    def paint(self, surface):
        print(self.name, surface)


red = Color()
red.name = "красный"

red.paint("стена")


## __init__


Вручную добавлять атрибуты после создания объекта неудобно. Можно забыть поле или написать имя с ошибкой.

Специальный метод `__init__` автоматически вызывается после создания нового объекта. Обычно в нём задают начальное состояние. Его еще называют метод-конструктор:


In [ ]:
class Expense:
    def __init__(self, title, amount, category):
        self.title = title
        self.amount = amount
        self.category = category


coffee = Expense("Кофе", 8.5, "Еда")
taxi = Expense("Такси", 24.0, "Транспорт")

print(coffee.title)
print(taxi.title)


Разберём строку:

```python
self.title = title
```

Справа `title` — параметр, полученный при вызове `Expense("Кофе", ...)`.

Слева `self.title` — атрибут, который мы сохраняем внутри нового объекта.

Имена параметров могут отличаться, но одинаковые имена параметров и атрибутов показывают связь понятнее.


In [ ]:
class Expense:
    def __init__(self, incoming_title: str, incoming_amount: float):
        self.title = incoming_title
        self.amount = incoming_amount


coffee = Expense("Кофе", 8.5)
print(coffee.title)


Добавим поведение. Метод может читать и изменять атрибуты текущего объекта.


In [ ]:
class Expense:
    def __init__(self, title: str, amount: float, category: str):
        self.title = title
        self.amount = amount
        self.category = category

    def describe(self) -> str:
        return f"{self.title}: {self.amount:.2f} BYN ({self.category})"

    def apply_discount(self, percent: float) -> None:
        self.amount *= 1 - percent / 100


coffee = Expense("Кофе", 10.0, "Еда")
print(coffee.describe())

coffee.apply_discount(20)
print(coffee.describe())


Состояние объекта изменилось. Переменная `coffee` осталась той же, но значение `coffee.amount` стало другим.

Метод `apply_discount` ничего не возвращает, поэтому его результат — `None`.


In [ ]:
result = coffee.apply_discount(10)
print(result)
print(coffee.amount)


Объект не должен появляться в заведомо неправильном состоянии. Проверку можно выполнить в `__init__`.

In [ ]:
class InvalidAmountError(ValueError):
    pass


class Expense:
    def __init__(self, title: str, amount: float):
        if amount <= 0:
            raise InvalidAmountError("Сумма расхода должна быть больше нуля")
        self.title = title
        self.amount = amount


try:
    impossible_expense = Expense("Возврат", -10)
except InvalidAmountError as error:
    print(error)


## Атрибуты класса и объекта

Атрибут экземпляра принадлежит отдельному объекту. Атрибут класса общий для всех объектов класса.


In [ ]:
class Expense:
    currency = "BYN"  # Атрибут класса

    def __init__(self, title: str, amount: float):
        self.title = title      # Атрибут объекта
        self.amount = amount    # Атрибут объекта


coffee = Expense("Кофе", 8.5)
taxi = Expense("Такси", 24.0)

print(coffee.currency)
print(taxi.currency)

Expense.currency = "USD"

print(coffee.currency)
print(taxi.currency)


Если значение описывает конкретный объект, сохраняйте его через `self`.

Если значение одинаково для всех объектов и относится к самому классу, можно использовать атрибут класса.

Попробуем создать класс `BankAccount`.

Объект должен хранить имя владельца и баланс. Добавим методы:

* `deposit(amount)` — пополняет баланс;
* `withdraw(amount)` — снимает деньги;
* `show_balance()` — возвращает строку с балансом.

При попытке снять больше денег, чем есть на счёте, поднимите `ValueError`.


In [ ]:
class BankAccount:
    # Реализуйте класс здесь.
    pass


account = BankAccount("Анна", 100)
account.deposit(50)
account.withdraw(30)
print(account.show_balance())

## Специальные методы

Имена вида `__name__` называют *специальными* или *магическими методами*. По-английски - *dunder* (от double under) *methods*.

Python вызывает их в определённых ситуациях. Например, `__init__` вызывается при инициализации объекта.

По умолчанию объект выводится не очень информативно:


In [ ]:
class Expense:
    def __init__(self, title: str, amount: float):
        self.title = title
        self.amount = amount


coffee = Expense("Кофе", 8.5)
print(coffee)


Метод `__repr__` возвращает строковое представление объекта, полезное разработчику.


In [ ]:
class Expense:
    def __init__(self, title: str, amount: float):
        self.title = title
        self.amount = amount

    def __repr__(self) -> str:
        return f"Expense(title={self.title!r}, amount={self.amount!r})"


coffee = Expense("Кофе", 8.5)
print(coffee)


Без `__eq__` два объекта сравниваются по идентичности. Одинаковые атрибуты сами по себе не делают объекты равными.


In [ ]:
first = Expense("Кофе", 8.5)
second = Expense("Кофе", 8.5)

print(first == second)
print(first is second)


Метод `__eq__` позволяет сравнивать классы по нашим правилам:

In [ ]:
class Expense:
    def __init__(self, title: str, amount: float):
        self.title = title
        self.amount = amount

    def __repr__(self) -> str:
        return f"Expense(title={self.title!r}, amount={self.amount!r})"

    def __eq__(self, other: 'Expense') -> bool:
        return self.title == other.title and self.amount == other.amount

first = Expense("Кофе", 8.5)
second = Expense("Кофе", 8.5)

print(first == second)
print(first is second)

## dataclasses

В Python есть способ автоматически настроить поведение класса без магических методов.

Для этого используется модуль `dataclasses` и специальный синтаксис - декоратор (`@dataclass`):


In [ ]:
from dataclasses import dataclass


@dataclass
class Expense:
    title: str
    amount: float
    category: str

coffee = Expense("Кофе", 8.5, "Еда")
another_coffee = Expense("Кофе", 8.5, "Еда")

print(coffee)
print(coffee == another_coffee)



Декоратор `@dataclass` автоматически создаёт шаблонные методы на основе аннотаций атрибутов:

* `__init__`;
* `__repr__`;
* `__eq__`.

Методы предметной области мы продолжаем писать самостоятельно.

`@dataclass` изменяет класс сразу после его объявления и добавляет методы, которые иначе пришлось бы писать вручную.

Обычные методы работают точно так же. `self` не меняет своего смысла.


In [ ]:
@dataclass
class Expense:
    title: str
    amount: float
    category: str

    def apply_discount(self, percent: float) -> None:
        self.amount *= 1 - percent / 100


coffee = Expense("Кофе", 10.0, "Еда")
coffee.apply_discount(20)
print(coffee.describe())


Аннотации не проверяют значения. `dataclass` использует аннотации для генерации методов, но не проверяет типы во время выполнения:

In [ ]:
strange_expense = Expense("Кофе", "дорого", "Еда")
print(strange_expense)


Если проверка нужна, используем метод `__post_init__`. Он вызывается сразу после сгенерированного `__init__`.


In [ ]:
@dataclass
class Expense:
    title: str
    amount: float
    category: str

    def __post_init__(self) -> None:
        if not self.title.strip():
            raise ValueError("Название не должно быть пустым")
        if self.amount <= 0:
            raise ValueError("Сумма должна быть больше нуля")


coffee = Expense("Кофе", 8.5, "Еда")
print(coffee)


Как и в функциях, поля со значениями по умолчанию записывают после обязательных полей.


In [ ]:
@dataclass
class Expense:
    title: str
    amount: float
    category: str = "Другое"
    paid: bool = False


coffee = Expense("Кофе", 8.5, "Еда")
book = Expense("Книга", 35.0)

print(coffee)
print(book)


В обычном классе мы передадим значения по умолчанию в `__init__()`:

In [ ]:
class Expense:
    def __init__(
        self,
        title: str,
        amount: float,
        category: str = "Другое",
        paid: bool = False
    ):
        pass

Помните про проблемы с изменяемыми значениями в качестве значений по умолчанию?

In [ ]:
@dataclass
class Report:
    title: str
    prices: list[int] = []

january = Report("Январь")
february = Report("Февраль")

january.prices.append(8.5)

print(january.prices)
print(february.prices)


Один общий список для всех объектов приводит к ошибкам. У датаклассов есть способ создавать пустой список(словарь, множество и т.д.) при помощи `field(default_factory=list)` для каждого нового объекта:


In [ ]:
from dataclasses import field


@dataclass
class Report:
    title: str
    prices: list[int] = field(default_factory=list)  # аргумент - функция без параметров, например lambda: SomeEmptyClass()


january = Report("Январь")
february = Report("Февраль")

january.prices.append(8.5)

print(january.prices)
print(february.prices)


Попробуем создать класс `Product` при помощи `@dataclass` с полями `name`, `price` и `quantity` (со значением по умолчанию).

Добавим проверку положительной цены и количества через `__post_init__`. Метод `total()` должен возвращать общую стоимость.


In [ ]:
@dataclass
class Product:
    # Добавьте поля и методы.
    pass

product = Product(name="Nike", price=99.50)
assert product.total() == 99.50


product = Product(name="Adidas", price=85.00, quantity=2)
assert product.total() == 170.00

try:
    product = Product(name="Reebok", price=83.45, quantity=-1)
except ValueError:
    pass

## property


Иногда нам нужен такой атрибут, который пересчитыват значения при каждом вызове.

`property` оставляет вычисление внутри класса, а снаружи позволяет обращаться к результату без круглых скобок.


In [ ]:
@dataclass
class Purchase:
    product: str
    price: float
    quantity: int = 1

    @property
    def total(self) -> float:
        return self.price * self.quantity


purchase = Purchase("Кофе", 18.5, 2)
print(purchase.total)  # Не purchase.total()


`total` зависит от других полей, поэтому его не нужно хранить отдельно. При изменении цены или количества свойство вернёт новое значение.


In [ ]:
purchase.quantity = 3
print(purchase.total)


## Композиция



Композиция означает: один объект содержит другие объекты.

Отчёт содержит расходы. Поэтому `Report` хранит список объектов `Expense`.


In [ ]:
@dataclass
class Expense:
    title: str
    amount: float
    category: str = "Другое"


@dataclass
class Report:
    title: str
    expenses: list[Expense] = field(default_factory=list)

    def add(self, expense: Expense) -> None:
        self.expenses.append(expense)

    @property
    def total(self) -> float:
        return sum(expense.amount for expense in self.expenses)


report = Report("Октябрь")
report.add(Expense("Кофе", 8.5, "Еда"))
report.add(Expense("Такси", 24.0, "Транспорт"))

print(report.expenses)
print(report.total)


Метод `.add()` - пример **инкапсуляции**. Мы не хотим, чтобы с атрибутами нашего класса взаимодействовали напрямую через методы, например, списка.

Вместо этого мы предоставляем способ работы с нашим классом - публичный интерфейс - в виде метода `.add()`. Это сильно поможет нам в будущем, когда мы захотим ввести новые правила:

* добавлять не больше 10 затрат в один отчет
* добавлять затраты на общую сумму до 100
* добавлять затраты только одной категории

и так далее.

Все эти детали реализации должны быть скрыты от внешнего кода и меняться **внутри** метода.


## Наследование


Наследование описывает отношение "является разновидностью".

Скажем в Python `bool` - разновидность `int`, но ограниченная двумя значениями:


In [ ]:
print(True + True / True)
print(type(True))
print(isinstance(True, int))

В данном случае `int` является **родительским** классом, а `bool` - **дочерним**. `bool` **наследует** поведение `int` - умеет делать все то же, что и `int`, плюс то, что умеет только сам `bool`.
 
Рассмотрим наш класс подписки:

In [ ]:
@dataclass
class Expense:
    title: str
    amount: float

    def describe(self) -> str:
        return f"{self.title}: {self.amount:.2f} BYN"

Разновидность расхода - подписка:

In [ ]:
@dataclass
class Subscription(Expense):
    period: str = "месяц"

music = Subscription("Музыка", 9.99)
print(music.describe())


В круглых скобках мы указали родительский класс для `Subscription`. В итоге дочерний класс `Subscription` получает поля (`title`, `amount`) и методы (`.describe()`) родительского класса.

Нам не до конца подходит то, как работает метод `.describe()` - нет информаци о том, что платеж осуществляется ежемесячно. Мы можем изменить - **переопределить** - метод:


In [ ]:
@dataclass
class Subscription(Expense):
    period: str = "месяц"

    def describe(self) -> str:
        return f"{self.title}: {self.amount:.2f} BYN / {self.period}"

music = Subscription("Музыка", 9.99)
print(music.describe())

Часть кода дублируется - это нехорошо. Что, если мы захотим изменить формат того, как мы описываем затрату? Придется менять код и в `Expense`, и в `Subscription`?

Мы можем вызвать родительский метод внутри дочернего при помощи функции `super()`:

In [ ]:
@dataclass
class Subscription(Expense):
    period: str = "месяц"

    def describe(self) -> str:
        desription = super().describe()
        return f"{desription} / {self.period}"

music = Subscription("Музыка", 9.99)
print(music.describe())

`super()` даёт доступ к реализации родительского класса. Здесь мы используем готовый `Expense.describe()` и расширяем результат.

Наследование полезно, когда дочерний объект действительно можно считать разновидностью родительского. Для связи между двумя классами типа "`A` содержит `B`" обычно лучше композиция.

## Классовый метод и cls


Иногда нужно создать объект другим способом. Например, из словаря после чтения JSON.

Декоратор `@classmethod` передаёт первым аргументом не объект, а класс. Этот параметр принято называть `cls`.


In [ ]:
@dataclass
class Expense:
    title: str
    amount: float
    category: str = "Другое"

    @classmethod
    def from_dict(cls, data: dict) -> "Expense":
        return cls(
            title=data["title"],
            amount=float(data["amount"]),
            category=data.get("category", "Другое"),
        )


data = {"title": "Кофе", "amount": 8.5, "category": "Еда"}
coffee = Expense.from_dict(data)
print(coffee)


Сравним первые параметры:

| Вид метода | Первый параметр | Что передаётся автоматически |
|---|---|---|
| обычный метод | `self` | конкретный объект |
| классовый метод | `cls` | класс |

`self` нужен для работы с состоянием объекта. `cls` полезен для альтернативного создания объектов и операций над классом.
